# 房價預測實戰：從數據到模型

## 📚 學習目標

本 notebook 將帶你完成一個完整的房價預測項目，包括：

1. **數據載入與探索**：了解數據特徵和分布
2. **數據預處理**：處理缺失值、異常值和特徵工程
3. **特徵標準化**：提升模型訓練效果
4. **模型訓練**：從零實現和框架實現對比
5. **模型評估**：使用多種指標評估性能
6. **結果視覺化**：直觀展示預測效果
7. **AI 輔助調優**：使用自動化工具優化模型

---

## 1. 環境設置

首先導入必要的庫並設置隨機種子以確保結果可重現。

In [ ]:
%matplotlib inline
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

# 導入我們的工具模組
from utils import (
    StandardScaler, MinMaxScaler, r_squared, 
    mean_absolute_error, root_mean_squared_error,
    set_random_seed, Timer
)

# 設置隨機種子
set_random_seed(42)

# 設置繪圖風格
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✓ 環境設置完成")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. 生成合成房價數據

為了教學目的，我們生成一個模擬的房價數據集。在實際項目中，你可以替換為真實數據（如波士頓房價數據集或加州房價數據集）。

### 特徵說明

| 特徵名稱 | 說明 | 單位 |
|---------|------|------|
| Area | 房屋面積 | 平方米 |
| Bedrooms | 臥室數量 | 個 |
| Age | 房齡 | 年 |
| Distance | 距市中心距離 | 公里 |
| Price | 房價（目標變量） | 萬元 |

In [ ]:
def generate_house_data(num_samples=1000, noise_level=0.1):
    """
    生成模擬房價數據
    
    Price = 10 + 0.3*Area + 5*Bedrooms - 0.5*Age - 2*Distance + noise
    """
    # 生成特徵
    area = np.random.uniform(50, 200, num_samples)  # 50-200平方米
    bedrooms = np.random.randint(1, 6, num_samples)  # 1-5個臥室
    age = np.random.uniform(0, 30, num_samples)     # 0-30年房齡
    distance = np.random.uniform(1, 20, num_samples) # 1-20公里
    
    # 計算價格（真實關係）
    price = (10 + 
            0.3 * area + 
            5 * bedrooms - 
            0.5 * age - 
            2 * distance)
    
    # 添加噪聲
    noise = np.random.normal(0, noise_level * price.std(), num_samples)
    price += noise
    
    # 創建DataFrame
    data = pd.DataFrame({
        'Area': area,
        'Bedrooms': bedrooms,
        'Age': age,
        'Distance': distance,
        'Price': price
    })
    
    return data

# 生成數據
df = generate_house_data(num_samples=1000)

print("數據生成完成！")
print(f"\n數據集大小: {df.shape}")
print(f"\n前5行數據:")
print(df.head())
print(f"\n數據統計信息:")
print(df.describe())

## 3. 數據探索與可視化

在訓練模型之前，我們需要了解數據的分布和特徵之間的關係。

In [ ]:
# 繪製特徵分布
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('特徵分布與相關性分析', fontsize=16, y=1.02)

# 繪製各特徵的直方圖
for idx, col in enumerate(df.columns):
    row = idx // 3
    col_idx = idx % 3
    axes[row, col_idx].hist(df[col], bins=30, alpha=0.7, edgecolor='black')
    axes[row, col_idx].set_xlabel(col)
    axes[row, col_idx].set_ylabel('Frequency')
    axes[row, col_idx].set_title(f'{col} Distribution')
    axes[row, col_idx].grid(True, alpha=0.3)

# 刪除多餘的子圖
fig.delaxes(axes[1, 2])

plt.tight_layout()
plt.show()

print("✓ 特徵分布圖繪製完成")

In [ ]:
# 相關性矩陣
plt.figure(figsize=(10, 8))
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', 
           cmap='coolwarm', center=0, 
           square=True, linewidths=1)
plt.title('Feature Correlation Matrix', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

print("\n特徵與價格的相關係數:")
print(correlation_matrix['Price'].sort_values(ascending=False))

In [ ]:
# 特徵與目標變量的散點圖
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Features vs Price', fontsize=16, y=1.01)

features = ['Area', 'Bedrooms', 'Age', 'Distance']
for idx, feature in enumerate(features):
    row = idx // 2
    col = idx % 2
    
    axes[row, col].scatter(df[feature], df['Price'], alpha=0.5, s=20)
    axes[row, col].set_xlabel(feature, fontsize=12)
    axes[row, col].set_ylabel('Price (萬元)', fontsize=12)
    axes[row, col].set_title(f'{feature} vs Price', fontsize=12)
    axes[row, col].grid(True, alpha=0.3)
    
    # 添加趨勢線
    z = np.polyfit(df[feature], df['Price'], 1)
    p = np.poly1d(z)
    axes[row, col].plot(df[feature], p(df[feature]), 
                       "r--", alpha=0.8, linewidth=2, label='Trend')
    axes[row, col].legend()

plt.tight_layout()
plt.show()

print("✓ 散點圖繪製完成")

## 4. 數據預處理

### 4.1 分割訓練集和測試集

In [ ]:
# 分離特徵和標籤
X = df[['Area', 'Bedrooms', 'Age', 'Distance']].values
y = df['Price'].values.reshape(-1, 1)

# 轉換為 PyTorch 張量
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

# 分割數據集（80% 訓練，20% 測試）
n = len(X_tensor)
n_train = int(0.8 * n)

indices = torch.randperm(n)
train_indices = indices[:n_train]
test_indices = indices[n_train:]

X_train, X_test = X_tensor[train_indices], X_tensor[test_indices]
y_train, y_test = y_tensor[train_indices], y_tensor[test_indices]

print(f"訓練集大小: {X_train.shape}")
print(f"測試集大小: {X_test.shape}")
print(f"\n訓練集標籤統計:")
print(f"  均值: {y_train.mean():.2f} 萬元")
print(f"  標準差: {y_train.std():.2f} 萬元")
print(f"  最小值: {y_train.min():.2f} 萬元")
print(f"  最大值: {y_train.max():.2f} 萬元")

### 4.2 特徵標準化

標準化可以：
- 加速梯度下降收斂
- 防止某些特徵主導學習過程
- 提高數值穩定性

In [ ]:
# 使用標準化器
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("特徵標準化完成！")
print(f"\n標準化後的訓練集統計:")
print(f"  均值: {X_train_scaled.mean(dim=0)}")
print(f"  標準差: {X_train_scaled.std(dim=0)}")

# 可視化標準化前後的對比
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('特徵標準化前後對比', fontsize=16)

feature_names = ['Area', 'Bedrooms', 'Age', 'Distance']

for i in range(4):
    # 標準化前
    axes[0, i].hist(X_train[:, i].numpy(), bins=30, alpha=0.7, edgecolor='black')
    axes[0, i].set_title(f'{feature_names[i]} (Original)', fontsize=11)
    axes[0, i].set_ylabel('Frequency')
    axes[0, i].grid(True, alpha=0.3)
    
    # 標準化後
    axes[1, i].hist(X_train_scaled[:, i].numpy(), bins=30, alpha=0.7, edgecolor='black')
    axes[1, i].set_title(f'{feature_names[i]} (Scaled)', fontsize=11)
    axes[1, i].set_ylabel('Frequency')
    axes[1, i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ 標準化對比圖繪製完成")

## 5. 模型訓練（從零實現）

首先使用從零實現的方式來訓練模型，深入理解每個步驟。

In [ ]:
# 初始化參數
def init_params(num_features):
    """初始化權重和偏置"""
    w = torch.normal(0, 0.01, size=(num_features, 1), requires_grad=True)
    b = torch.zeros(1, requires_grad=True)
    return w, b

# 定義模型
def linreg(X, w, b):
    """線性回歸模型"""
    return torch.matmul(X, w) + b

# 定義損失函數
def squared_loss(y_hat, y):
    """均方損失"""
    return ((y_hat - y.reshape(y_hat.shape)) ** 2 / 2).mean()

# 定義優化器
def sgd(params, lr):
    """隨機梯度下降"""
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad
            param.grad.zero_()

print("✓ 模型組件定義完成")

In [ ]:
# 訓練配置
num_features = X_train_scaled.shape[1]
lr = 0.01
num_epochs = 100
batch_size = 32

# 初始化參數
w, b = init_params(num_features)

# 創建數據加載器
train_dataset = TensorDataset(X_train_scaled, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# 記錄訓練過程
train_losses = []
test_losses = []

# 開始訓練
print("開始訓練（從零實現）...")
timer = Timer()

for epoch in range(num_epochs):
    for X_batch, y_batch in train_loader:
        # 前向傳播
        y_hat = linreg(X_batch, w, b)
        loss = squared_loss(y_hat, y_batch)
        
        # 反向傳播
        loss.backward()
        
        # 參數更新
        sgd([w, b], lr)
    
    # 記錄損失
    with torch.no_grad():
        train_loss = squared_loss(linreg(X_train_scaled, w, b), y_train)
        test_loss = squared_loss(linreg(X_test_scaled, w, b), y_test)
        train_losses.append(train_loss.item())
        test_losses.append(test_loss.item())
    
    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1:3d}: Train Loss = {train_loss:.4f}, Test Loss = {test_loss:.4f}')

training_time = timer.stop()
print(f"\n訓練完成！總耗時: {training_time:.2f} 秒")
print(f"\n學習到的參數:")
print(f"權重 w: {w.detach().numpy().flatten()}")
print(f"偏置 b: {b.item():.4f}")

In [ ]:
# 繪製訓練過程
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Train Loss', linewidth=2)
plt.plot(test_losses, label='Test Loss', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training Progress (From Scratch)', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✓ 訓練曲線繪製完成")

## 6. 模型訓練（PyTorch 框架）

使用 PyTorch 的高級 API 重新訓練模型，對比實現難度和效果。

In [ ]:
# 定義模型
class LinearRegressionModel(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.linear = nn.Linear(num_features, 1)
    
    def forward(self, x):
        return self.linear(x)

# 創建模型
model = LinearRegressionModel(num_features)

# 定義損失函數和優化器
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=lr)

# 記錄訓練過程
framework_train_losses = []
framework_test_losses = []

# 開始訓練
print("開始訓練（PyTorch 框架）...")
timer = Timer()

for epoch in range(num_epochs):
    model.train()
    for X_batch, y_batch in train_loader:
        # 前向傳播
        y_hat = model(X_batch)
        loss = criterion(y_hat, y_batch)
        
        # 反向傳播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # 評估
    model.eval()
    with torch.no_grad():
        train_loss = criterion(model(X_train_scaled), y_train)
        test_loss = criterion(model(X_test_scaled), y_test)
        framework_train_losses.append(train_loss.item())
        framework_test_losses.append(test_loss.item())
    
    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1:3d}: Train Loss = {train_loss:.4f}, Test Loss = {test_loss:.4f}')

framework_training_time = timer.stop()
print(f"\n訓練完成！總耗時: {framework_training_time:.2f} 秒")

# 打印學習到的參數
print(f"\n學習到的參數:")
with torch.no_grad():
    print(f"權重 w: {model.linear.weight.numpy().flatten()}")
    print(f"偏置 b: {model.linear.bias.item():.4f}")

In [ ]:
# 對比兩種實現方式的訓練曲線
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 從零實現
axes[0].plot(train_losses, label='Train Loss', linewidth=2)
axes[0].plot(test_losses, label='Test Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('From Scratch', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# PyTorch 框架
axes[1].plot(framework_train_losses, label='Train Loss', linewidth=2)
axes[1].plot(framework_test_losses, label='Test Loss', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('PyTorch Framework', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ 對比圖繪製完成")

## 7. 模型評估

使用多種評估指標來全面評估模型性能。

In [ ]:
# 使用框架訓練的模型進行預測
model.eval()
with torch.no_grad():
    y_train_pred = model(X_train_scaled)
    y_test_pred = model(X_test_scaled)

# 計算評估指標
train_r2 = r_squared(y_train_pred, y_train)
test_r2 = r_squared(y_test_pred, y_test)

train_mae = mean_absolute_error(y_train_pred, y_train)
test_mae = mean_absolute_error(y_test_pred, y_test)

train_rmse = root_mean_squared_error(y_train_pred, y_train)
test_rmse = root_mean_squared_error(y_test_pred, y_test)

# 打印評估結果
print("="*60)
print("模型評估結果")
print("="*60)
print(f"{'指標':<15s} {'訓練集':<20s} {'測試集':<20s}")
print("-"*60)
print(f"{'R² Score':<15s} {train_r2:<20.4f} {test_r2:<20.4f}")
print(f"{'MAE':<15s} {train_mae:<20.4f} {test_mae:<20.4f}")
print(f"{'RMSE':<15s} {train_rmse:<20.4f} {test_rmse:<20.4f}")
print("="*60)

# 判斷模型狀態
if abs(train_r2 - test_r2) < 0.05:
    print("\n✓ 模型狀態良好（訓練集和測試集性能接近）")
elif train_r2 > test_r2 + 0.1:
    print("\n⚠ 可能存在過擬合（訓練集性能明顯優於測試集）")
else:
    print("\n⚠ 可能存在欠擬合（整體性能較低）")

## 8. 結果可視化

In [ ]:
# 預測值 vs 真實值
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 訓練集
axes[0].scatter(y_train.numpy(), y_train_pred.numpy(), alpha=0.5, s=30)
axes[0].plot([y_train.min(), y_train.max()], 
            [y_train.min(), y_train.max()], 
            'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('True Price (萬元)', fontsize=12)
axes[0].set_ylabel('Predicted Price (萬元)', fontsize=12)
axes[0].set_title(f'Training Set (R² = {train_r2:.4f})', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# 測試集
axes[1].scatter(y_test.numpy(), y_test_pred.numpy(), alpha=0.5, s=30)
axes[1].plot([y_test.min(), y_test.max()], 
            [y_test.min(), y_test.max()], 
            'r--', linewidth=2, label='Perfect Prediction')
axes[1].set_xlabel('True Price (萬元)', fontsize=12)
axes[1].set_ylabel('Predicted Price (萬元)', fontsize=12)
axes[1].set_title(f'Test Set (R² = {test_r2:.4f})', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ 預測對比圖繪製完成")

In [ ]:
# 殘差分析
residuals_train = (y_train - y_train_pred).numpy()
residuals_test = (y_test - y_test_pred).numpy()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('殘差分析', fontsize=16, y=1.01)

# 訓練集殘差分布
axes[0, 0].hist(residuals_train, bins=30, alpha=0.7, edgecolor='black')
axes[0, 0].set_xlabel('Residual', fontsize=11)
axes[0, 0].set_ylabel('Frequency', fontsize=11)
axes[0, 0].set_title('Training Set Residuals Distribution', fontsize=12)
axes[0, 0].axvline(0, color='r', linestyle='--', linewidth=2)
axes[0, 0].grid(True, alpha=0.3)

# 測試集殘差分布
axes[0, 1].hist(residuals_test, bins=30, alpha=0.7, edgecolor='black')
axes[0, 1].set_xlabel('Residual', fontsize=11)
axes[0, 1].set_ylabel('Frequency', fontsize=11)
axes[0, 1].set_title('Test Set Residuals Distribution', fontsize=12)
axes[0, 1].axvline(0, color='r', linestyle='--', linewidth=2)
axes[0, 1].grid(True, alpha=0.3)

# 訓練集殘差散點圖
axes[1, 0].scatter(y_train_pred.numpy(), residuals_train, alpha=0.5, s=20)
axes[1, 0].axhline(0, color='r', linestyle='--', linewidth=2)
axes[1, 0].set_xlabel('Predicted Price', fontsize=11)
axes[1, 0].set_ylabel('Residual', fontsize=11)
axes[1, 0].set_title('Training Set Residuals vs Predictions', fontsize=12)
axes[1, 0].grid(True, alpha=0.3)

# 測試集殘差散點圖
axes[1, 1].scatter(y_test_pred.numpy(), residuals_test, alpha=0.5, s=20)
axes[1, 1].axhline(0, color='r', linestyle='--', linewidth=2)
axes[1, 1].set_xlabel('Predicted Price', fontsize=11)
axes[1, 1].set_ylabel('Residual', fontsize=11)
axes[1, 1].set_title('Test Set Residuals vs Predictions', fontsize=12)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ 殘差分析圖繪製完成")
print(f"\n殘差統計:")
print(f"訓練集殘差均值: {residuals_train.mean():.4f}")
print(f"測試集殘差均值: {residuals_test.mean():.4f}")
print(f"訓練集殘差標準差: {residuals_train.std():.4f}")
print(f"測試集殘差標準差: {residuals_test.std():.4f}")

## 9. 特徵重要性分析

分析各個特徵對房價的影響程度。

In [ ]:
# 提取權重
feature_names = ['Area', 'Bedrooms', 'Age', 'Distance']
weights = model.linear.weight.detach().numpy().flatten()

# 繪製特徵重要性
plt.figure(figsize=(10, 6))
colors = ['green' if w > 0 else 'red' for w in weights]
plt.barh(feature_names, weights, color=colors, alpha=0.7, edgecolor='black')
plt.xlabel('Weight', fontsize=13)
plt.title('Feature Importance (Model Weights)', fontsize=14)
plt.axvline(0, color='black', linewidth=1)
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("特徵權重分析:")
print("="*50)
for name, weight in zip(feature_names, weights):
    direction = "正相關" if weight > 0 else "負相關"
    print(f"{name:<12s}: {weight:>8.4f} ({direction})")
print("="*50)

print("\n解讀:")
print("- 權重為正：特徵增加時，房價上升")
print("- 權重為負：特徵增加時，房價下降")
print("- 權重絕對值越大，影響越顯著")

## 10. 模型應用：實際預測

使用訓練好的模型進行實際預測。

In [ ]:
def predict_house_price(area, bedrooms, age, distance, model, scaler):
    """
    預測房價
    
    參數:
        area: 面積(平方米)
        bedrooms: 臥室數量
        age: 房齡(年)
        distance: 距市中心距離(公里)
        model: 訓練好的模型
        scaler: 數據標準化器
    
    返回:
        predicted_price: 預測價格(萬元)
    """
    # 構造輸入
    X_new = torch.tensor([[area, bedrooms, age, distance]], dtype=torch.float32)
    
    # 標準化
    X_new_scaled = scaler.transform(X_new)
    
    # 預測
    model.eval()
    with torch.no_grad():
        price = model(X_new_scaled)
    
    return price.item()

# 測試案例
test_cases = [
    {"area": 100, "bedrooms": 3, "age": 5, "distance": 10, "desc": "中等面積，新房，距離適中"},
    {"area": 150, "bedrooms": 4, "age": 2, "distance": 5, "desc": "大面積，較新，市中心附近"},
    {"area": 60, "bedrooms": 2, "age": 20, "distance": 15, "desc": "小面積，老房，郊區"},
]

print("="*80)
print("房價預測示例")
print("="*80)

for i, case in enumerate(test_cases, 1):
    price = predict_house_price(
        case['area'], case['bedrooms'], 
        case['age'], case['distance'],
        model, scaler
    )
    
    print(f"\n案例 {i}: {case['desc']}")
    print(f"  - 面積: {case['area']} 平方米")
    print(f"  - 臥室: {case['bedrooms']} 個")
    print(f"  - 房齡: {case['age']} 年")
    print(f"  - 距離: {case['distance']} 公里")
    print(f"  ➜ 預測價格: {price:.2f} 萬元")

print("\n" + "="*80)

## 11. AI 輔助：超參數調優建議

### 💡 如何使用 AI 工具改進模型

你可以向 AI 助手詢問以下問題：

1. **"如何選擇最佳學習率？"**
   - AI 會建議使用學習率調度器或自動調參工具

2. **"我的模型過擬合了，怎麼辦？"**
   - AI 會推薦正則化方法（L1/L2）或增加數據

3. **"特徵工程有什麼建議？"**
   - AI 會分析特徵相關性，建議特徵組合或轉換

4. **"如何提高模型性能？"**
   - AI 會從多個角度分析並提供改進方案

### 🤖 AI 輔助工作流

```python
# 1. 準備問題描述
problem_description = """
我正在做房價預測，使用線性回歸模型。
當前 R² = 0.85，但測試集比訓練集低 0.1。
請幫我分析問題並提供改進建議。
"""

# 2. 與 AI 對話
# 將問題描述複製給 ChatGPT/Claude/Gemini 等

# 3. 根據 AI 建議實驗
# 例如：添加正則化、調整學習率、特徵工程等
```

## 12. 總結與進階方向

### ✅ 本章節你學到了

1. 完整的機器學習項目流程
2. 數據探索和可視化的重要性
3. 特徵標準化對訓練的影響
4. 從零實現 vs 框架實現的對比
5. 多種評估指標的使用
6. 模型診斷和改進方法
7. 如何使用 AI 輔助開發

### 🚀 進階學習方向

1. **正則化** → 查看 `10_regularization.ipynb`
   - Ridge 回歸 (L2)
   - Lasso 回歸 (L1)
   - Elastic Net

2. **特徵工程** → 查看 `14_feature-engineering.ipynb`
   - 多項式特徵
   - 特徵選擇
   - 特徵交互

3. **模型評估** → 查看 `11_model-evaluation.ipynb`
   - 交叉驗證
   - 學習曲線
   - 模型比較

4. **AI 輔助開發** → 查看 `13_ai-assisted-ml.ipynb`
   - 自動超參數調優
   - 模型解釋性
   - AutoML 工具

### 💪 練習題

1. 嘗試使用真實數據集（如加州房價數據集）
2. 添加更多特徵並觀察影響
3. 實現不同的學習率調度策略
4. 比較不同優化器的效果（SGD, Adam, RMSprop）
5. 使用 AI 工具優化你的模型

---

**恭喜完成房價預測實戰！🎉**

_繼續探索更多進階內容..._